AI Travel Itinerary Generator using OpenAI, OpenWeatherMap, and Google Places API

Learning Objectives:

- Understand how to install and use the OpenAI API in Google Colab.
- Learn how to import necessary Python libraries for different tasks.
- Understand how to authenticate with the OpenAI API using an API key.
- Learn to work with lists and randomly select items using Python.
- Practice using an AI model to generate creative text based on a provided prompt.
- Gain familiarity with basic functions in Python and how to interact with APIs.


Technologies Used:  

- Google Colab: An online platform that allows running Python code easily without local setup.
- OpenAI GPT-4o-mini (gpt-4o-mini): A pre-trained language model used to generate creative text based on user input.

In [ ]:
# Install the required packages for Google Colab
# This will set up everything you need for your code.
!pip install openai requests python-dotenv

In [ ]:
# RAG++ (Retrieval-Augmented Generation) concept: Combines data retrieval from APIs with GPT generation to create a personalized output.
# GPT (OpenAI): Used to generate natural language itineraries.
# OpenWeatherMap API: To retrieve weather information for the destination.
# Google Places API: To retrieve information about popular places to visit.

# Step 1: Set Up Your Environment
# Import the necessary libraries
# We'll need these libraries to handle API requests, JSON data, and the OpenAI language model.
import requests # To handle API requests to get travel data
import json # To parse JSON data from the APIs
from openai import OpenAI # To interact with GPT models
import datetime # To handle date operations

- OPENWEATHERMAP: https://openweathermap.org/
- Google Places API: https://cloud.google.com

In [ ]:
import os
from openai import OpenAI

OPENWEATHERMAP_API_KEY = os.getenv("OPENWEATHERMAP_API_KEY")
GOOGLE_PLACES_API_KEY = os.getenv("GOOGLE_PLACES_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

client = OpenAI(api_key=OPENAI_API_KEY)


The Google Places API is a service provided by Google that allows developers to integrate location-based functionalities into their applications. It provides access to a wide range of data about places such as businesses, landmarks, and points of interest. Here's a breakdown of what it offers:

What Can the Google Places API Do?  

Search for Places:
- Perform text or keyword searches for places (e.g., "restaurants near me"). 
- Search within a specific geographic area using coordinates. 

Retrieve Place Details:  
- Access detailed information about a place, including its name, address, phone number, reviews, and photos.  

Autocomplete Search:  
- Enhance user experience by providing suggestions as users type in a search box.

Geocoding and Reverse Geocoding:  
- Convert addresses into geographic coordinates or vice versa.  

Nearby Places:  
- Find places within a given radius of a location (e.g., "cafes within 5 km").

Place Photos:  
- Fetch photos associated with a place, like images of landmarks or storefronts.

Place Reviews and Ratings:  
- Access user-generated reviews and ratings for a specific place.


In [ ]:
# Step 3: Get User Preferences
# Let's collect the user input to personalize the itinerary.
# We use input statements to gather user preferences such as destination, travel dates, activities, and budget.
print("Please enter your travel preferences below:") # Prompt user to enter travel preferences
destination = input("🌍 Destination: ") # Get the destination city from the user
dates = input("📅 Travel dates (e.g., 2024-12-01 to 2024-12-05) ") # Get the travel dates from the user
activities = input("🏃 Preferred activities (historical, art, beaches): ")  # Get preferred activities
budget = input("💸 Budget per day in CAD: ") # Get budget per day from the user

In [ ]:
# Step 4: Retrieve Data Using APIs
# Define a function to get weather data using OpenWeatherMap API
# The weather data will help us determine if certain activities are suitable.
def get_weather_data(city):
  # Construct the URL for the OpenWeatherMap API using the city name and API key
  url = f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={OPENWEATHERMAP_API_KEY}&units=metric"

  # Send a GET request to the API URL
  response = requests.get(url)

  # Check if the response is successful (status code 200)
  if response.status_code == 200:
       # Parse the response JSON and extract necessary weather information
      weather_data = response.json()
      return {
          "description": weather_data['weather'][0]['description'],
          "temperature": weather_data['main']['temp']
      }
  else:
      # Provide detailed error message based on the status code
      if response.status_code == 401:
        print("Error: Unauthorized. Please check your API key")
      elif response.status_code == 404:
        print(f"Error: City '{city}' not found. Double check the spelling of the city and re-enter the city name in Step 3.")
      elif response.status_code == 429:
        print("Error: Too many requests. You may have hit the limit rate for your API key.")
      else:
        print(f"Error: Unable to fetch weather data. Status code: {response.status_code}, Response: {response.text}")
      return None


In [ ]:
# Step 5: Get Data on Popular Places
# Use Google Places API to get information about popular tourist attractions.
# We'll retrieve this data and present it in the itinerary later.
def get_places_data(city, activity_type):
  # Define mappings for better results
  activity_mapping = { #dictionary
    "waffles": "cafes or dessert shops",
    "art": "art galleries or museums",
    "historical": "historical landmarks",
    "museum": "museums",
    "museums": "museums",
    "park": "parks",
    "parks": "parks",
    "cafe": "cafes",
    "cafes": "cafes",
    "beach": "beaches",
    "beaches": "beaches",
    "food": "restaurants",
    "shopping": "shopping malls",
    "nature": "nature attractions"
}
  # Map the user activity input to a more appropriate query
  mapped_activity = activity_mapping.get(activity_type.lower(), activity_type) #handling cases where input might not have exactly matched the dictionary key

  # Construct the URL for Google Places API using the city, activity type, and API key
  url = f"https://maps.googleapis.com/maps/api/place/textsearch/json?query={mapped_activity}+in+{city}&key={GOOGLE_PLACES_API_KEY}"
  # Send a GET request to the API URL
  response = requests.get(url)
    # Check if the response is successful (status code 200)
  if response.status_code == 200:
        # Parse the response JSON and extract relevant place information
      places_data = response.json()
      # Handle Google Places API errors
      if places_data.get("status") != "OK":
            print("Google Places API error:", places_data.get("status"))
            print("Message:", places_data.get("error_message"))
            return []
      places = []
      for place in places_data.get('results', [])[:5]: # Limit results to the top 5 places
        places.append({
            "name": place.get('name', 'No name available'), # Name of the place
            "address": place.get('formatted_address', 'No address available'), # Address of the place
            "rating": place.get('rating', 'No rating available') # Rating of the place, if available
          })
      return places # Return the list of places

  else:
    print(f"Error: Unable to fetch places data. Status code: {response.status_code}")
    return [] # Return [] if the request was unsuccessful

In [ ]:
# Step 6: Use GPT to Generate an Itinerary
# This is where we put it all together. We use GPT to generate a natural language itinerary using the gathered data.
def promptGPT(prompt, model, system_requirements):
    # Imagine GPT-4o-mini is your creative writing buddy. You give it a prompt, and it comes up with a cool story idea.
    response = client.chat.completions.create ( # The client.chat.completions.create function in the OpenAI API allows you to generate AI responses by providing a sequence of messages that simulate a conversation.
        model = model,
        messages = [
            {"role": "system", "content": system_requirements}, # Setting the personality of the assistant, like saying "You're a storyteller."
            {"role": "user", "content": prompt} # Providing the specific question or task, like a user asking to continue the story.
        ]
    )
    # Extracting the generated text from the response
    # Think of the response like a letter from GPT with the story written inside. We just need to open it and read it.
    return response.choices[0].message.content.strip()


In [ ]:
# Step 7: Get the Data for User Input
# Now we gather weather and places data using the functions we wrote earlier.
weather_info = get_weather_data(destination) # Get weather information for the destination
places_info = get_places_data(destination, activities) # Get information about popular places to visit


In [ ]:
# Step 8A: Basic Itinerary Generation
# Now, we bring it all together and generate a final itinerary for the user.
# Generate the travel itinerary based on user preferences, weather info, and places info
system_requirements = "You are a travel assistant who is very friendly and informative."
prompt = f"You are a travel guide. Create a detailed 3-day travel itinerary for {destination}. \n\n"
prompt += f"The traveler has a daily budget of ${budget} CAD.\n\n"
# Add weather information to the prompt if available
if weather_info and places_info:
  prompt += f"The weather in {destination} is currently {weather_info['description']} with a temperature of {weather_info['temperature']} degrees Celsius. \n\n"
    # Add information about places to visit
  prompt += "Places to visit include:\n"
  for place in places_info:
    prompt += f"- {place['name']}, located at {place['address']}, has a rating of {place['rating']}\n"
  prompt += "\nPlease generate a daily plan for a traveler."
    # Use promptGPT function to generate the itinerary
  itinerary = promptGPT(prompt, "gpt-4o-mini", system_requirements)
    # Print the personalized travel itinerary
  print("\n 📍 Your Personalized Travel Itinerary:\n")
  print(itinerary)
else:
    # Print an error message if any of the required data couldn't be retrieved
    if weather_info is None:
      print("Sorry, we're having trouble fetching the weather information. Please check the city name or your API key.")
    if not places_info:
      print("Sorry, we're having trouble fetching the places information. Please check the activity type or your API key.")


Going a bit further:
- The output is more contextually aware and responsive to the user's environment.
- Users get to know why certain activities are being suggested, which makes the itinerary more helpful and trustworthy.
- This feature makes your itinerary more practical and realistic—ensuring the user enjoys the trip no matter the weather! 

Key Enhancements:

Weather-Based Activity Filtering:
- The function is_weather_good_for_outdoors() checks the weather description and temperature to determine if it's suitable for outdoor activities.

Dynamic Activity Selection:
- When bad weather is detected, the generator filters out outdoor activities.
- This is done by using a conditional check to include only indoor activities when the weather is poor.

Itinerary Output with Recommendations:
- Each activity includes additional information to indicate why it was chosen:
- If it's an indoor activity and the weather is bad: "Recommended due to current weather conditions".
- If it's an outdoor activity and the weather is good: "Enjoyable given the current good weather conditions".

Dynamic Mapping of User Activities:
- A mapping dictionary (activity_type_mapping) is used to determine if an activity is "indoor" or "outdoor".
- This allows more accurate recommendations based on the weather.

In [ ]:
activity_type_mapping = {
    "waffle": "indoor",
    "scooter": "outdoor",
    "art": "indoor",
    "historical": "outdoor",
    "museum": "indoor",
    "park": "outdoor",
    "cafe": "indoor",
    "beach": "outdoor",
    "food": "indoor",
    "shopping": "indoor",
    "nature": "outdoor"
}

In [ ]:
# Determine whether the weather is suitable for outdoor activities
def is_weather_good_for_outdoors(weather_description, temperature):
    bad_weather_conditions = ["rain", "thunderstorm", "snow", "drizzle"]
    # If any bad condition is mentioned in the weather description, consider it bad weather
    if any(condition in weather_description.lower() for condition in bad_weather_conditions):
        return False
    # Consider temperatures below 5°C or above 35°C as unsuitable for outdoor activities
    if temperature < 5 or temperature > 35:
        return False
    return True

# Step 8B: Weather-Aware Itinerary Generation
# Now, we bring it all together and generate a final itinerary for the user.
if weather_info and places_info:
    # Determine if the weather is good for outdoor activities
    is_good_weather = is_weather_good_for_outdoors(weather_info['description'], weather_info['temperature'])

    # Generate the travel itinerary based on user preferences, weather info, and places info
    system_requirements = "You are a travel assistant who is very friendly and informative."
    prompt = f"You are a travel guide. Create a detailed 3-day travel itinerary for {destination}.\n\n"
    prompt += f"The traveler has a daily budget of ${budget} CAD.\n\n"

    # Add weather information to the prompt if available
    if weather_info:
        prompt += f"The weather in {destination} is currently {weather_info['description']} with a temperature of {weather_info['temperature']} degrees Celsius.\n\n"

    # Add information about places to visit based on weather suitability
    prompt += "Places to visit include:\n"
    for place in places_info:
        activity_name = place['name'].lower()
        activity_type = "unknown"

        for keyword, value in activity_type_mapping.items():
            if keyword in activity_name:
                activity_type = value
                break  # Check if the activity type in the mapping

        # Skip outdoor activities if the weather is bad
        if not is_good_weather and activity_type == "outdoor":
            continue

        # Add activity to the prompt
        prompt += f"- {place['name']}, located at {place['address']}, has a rating of {place['rating']}\n"
        # Indicate if it's recommended due to indoor/outdoor suitability
        if activity_type == "indoor" and not is_good_weather:
            prompt += "  (Recommended due to current weather conditions)\n"
        elif activity_type == "outdoor" and is_good_weather:
            prompt += "  (Enjoyable given the current good weather conditions)\n"

    prompt += "\nPlease generate a daily plan for the traveler, considering the suitability of each activity based on the weather conditions."

    # Use promptGPT function to generate the itinerary
    itinerary = promptGPT(prompt, "gpt-4o-mini", system_requirements)
    # Print the personalized travel itinerary
    print("\n📍 Your Personalized Travel Itinerary:\n")
    print(itinerary)
else:
    # Print an error message if any of the required data couldn't be retrieved
    if weather_info is None:
        print("Sorry, we're having trouble fetching the weather information. Please check the city name or your API key.")
    if not places_info:
        print("Sorry, we're having trouble fetching the places information. Please check the activity type or your API key.")

